In [48]:
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv()

client = Groq()
MODEL = "openai/gpt-oss-120b"


def llm(prompt: str) -> str:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content



In [49]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

Absolutely! You can enroll in the course at any time, provided enrollment is still open for the current session. Here’s a quick rundown of what you need to do:

### 1. Check the Enrollment Window
- **Open enrollment:** Most of our courses accept new students on a rolling basis, so you can usually sign up right away.
- **Closed enrollment:** If the current cohort is already full or the session has started, you may need to wait for the next opening. In that case, we can add you to the waitlist and notify you as soon as a spot opens.

### 2. Meet Any Prerequisites
- **Basic requirements:** Some courses have recommended background knowledge (e.g., introductory programming, basic statistics, or a specific software tool). If you meet those, you’re good to go.
- **No formal prerequisites:** Many of our introductory courses have no required prior experience—just enthusiasm and a willingness to learn!

### 3. Register
1. **Visit the enrollment page** on our website (or follow the link we’ll sen

In [50]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [51]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [52]:
print(prompt)


Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
I just discovered the course. Can I join now?

Context:

I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students partici

In [53]:
answer = llm(prompt)
print(answer)

Yes—you can still join the course. If you’d like to earn a certificate, just be sure to submit your project before the submission deadline (while we’re still accepting projects).


In [54]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [55]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [56]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1406

In [57]:
documents[0]

{'id': '0e38656cfb',
 'course': 'machine-learning-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'How do I submit homework?',
 'answer': "- Do the tasks locally\n- Publish your code (e.g., in your own GitHub repo)\n- Submit your answers via the homework form and include the URL to your code\n- You will see the answers only after the deadline\n- Homeworks are in the cohorts folder, e.g. for 2025 it's [`cohorts/2025`](https://github.com/DataTalksClub/machine-learning-zoomcamp/tree/master/cohorts/2025)\n- The forms for submitting the homework are in the [course management platform](https://courses.datatalks.club/)"}

In [58]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [59]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '5cc511f85b',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Does the course certificate show the number of course hours?',
  'answer': 'No. The certificate does not 

In [60]:
'''
Boosting field
Not all fields are equally important. The question field is usually more relevant than section for matching. Query words appearing in the question is a stronger signal than them appearing in the section name.
'''

results = index.search(
    question,
    num_results=5,
    boost_dict={"question": 2.0, "section": 0.5}
)

In [61]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '41aabbd7c5',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'},
 {'id': '2d8b16c2a0',
  'course': 'mlops-zoomcamp',
  'section':

In [62]:
'''
Filtering by course
Sometimes you want to restrict the search to a specific course.

minsearch supports keyword filtering:
'''
results = index.search(
    question,
    num_results=5,
    filter_dict={"course": "mlops-zoomcamp"}
)
results

[{'id': '2d8b16c2a0',
  'course': 'mlops-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course - Can I still join the course after the start date?',
  'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."},
 {'id': 'c842475338',
  'course': 'mlops-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Homework: Just found this course, can I still submit homeworks?',
  'answer': 'To clarify on **late homework submissions**:\n\n- You cannot submit after the homework is scored, as the form is closed.\n- Once the form is closed (i.e., scored), no further submissions are possible.\n- You can check your code against the solution by reviewing the `homework.md` file.\n\nIf the due date has passed but the form is still "O

In [63]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [64]:
search_results = search(question)
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '5cc511f85b',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Does the course certificate show the number of course hours?',
  'answer': 'No. The certificate does not 

In [65]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

"""

In [66]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [67]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [68]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [69]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Does the course certificate show the number of course hours?
A: No. The certificate does not state a total number of hours.

Module 3: Orchestration
Q: Why do we need orchestration / Kestra — can't I just run the code in a notebook?
A: Notebooks are great for learning and experimenting, but real AI w

In [70]:
# Groq
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": prompt}],
)
text = response.choices[0].message.content

In [71]:
response.choices[0]

Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Yes—you can hop in right away. You can start working through the material immediately (registration is only for gauging interest and isn’t required to begin).  \n\nIf you want a certificate, just make sure you submit your capstone project before the cohort’s submission window closes; otherwise you can still learn the content without a certificate.', role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning='We need to answer question: "I just discovered the course. Can I join now?" The answer is in the context: Yes, but need to submit project while still accepting submissions for certificate. So respond accordingly. Also perhaps mention you can start learning now, no registration needed. Provide brief answer.', tool_calls=None))

In [72]:
print(response.model_dump_json(indent=2))

{
  "id": "chatcmpl-1516f9f6-28f9-4bf8-830a-063a25bf76cf",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "Yes—you can hop in right away. You can start working through the material immediately (registration is only for gauging interest and isn’t required to begin).  \n\nIf you want a certificate, just make sure you submit your capstone project before the cohort’s submission window closes; otherwise you can still learn the content without a certificate.",
        "role": "assistant",
        "annotations": null,
        "executed_tools": null,
        "function_call": null,
        "reasoning": "We need to answer question: \"I just discovered the course. Can I join now?\" The answer is in the context: Yes, but need to submit project while still accepting submissions for certificate. So respond accordingly. Also perhaps mention you can start learning now, no registration needed. Provide brief answer.",
 

In [73]:
response.choices[0].message.content 

'Yes—you can hop in right away. You can start working through the material immediately (registration is only for gauging interest and isn’t required to begin).  \n\nIf you want a certificate, just make sure you submit your capstone project before the cohort’s submission window closes; otherwise you can still learn the content without a certificate.'

In [74]:
# Para saber quantos tokens foram usados, podemos acessar o atributo `usage` do objeto `response`. Isso nos dará informações sobre o número de tokens consumidos na solicitação e na resposta.
response.usage

CompletionUsage(completion_tokens=136, prompt_tokens=571, total_tokens=707, completion_time=0.285004553, completion_tokens_details=CompletionTokensDetails(reasoning_tokens=61), prompt_time=0.052993556, prompt_tokens_details=None, queue_time=0.180616792, total_time=0.337998109)

In [75]:
# preços por 1M tokens — confira em groq.com/pricing pro seu MODEL - No caso do nosso, é free, mas vamos calcular o custo para fins de aprendizado.
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.prompt_tokens * input_price
    + response.usage.completion_tokens * output_price
)

cost

0.00104025

In [76]:
message_history = [
    {"role": "system", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt},
]

response = client.chat.completions.create(
    model=MODEL,
    messages=message_history,
)

text = response.choices[0].message.content

In [78]:
def llm(instructions, prompt, model=MODEL):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": instructions},
            {"role": "user", "content": prompt},
        ],
    )
    return response.choices[0].message.content

In [79]:
def rag(query, model=MODEL):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [80]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes—you can still join the course. Just keep in mind that if you want to earn a certificate, you’ll need to submit your capstone project while the course is still accepting submissions.


In [ ]:
rag("How do I get a certificate?")

'You can earn a certificate only by completing the course as part of a **live cohort**.  \nHere’s what you need to do:\n\n1. **Finish the capstone project** – submit your final project.  \n2. **Complete all assigned peer reviews** – you’ll automatically receive review assignments after project submissions close, and you must finish them within the peer‑review window.  \n3. **(Optional) Homework** – completing homework is recommended but not required for the certificate.  \n\nYou may work through the course material and prepare your project in self‑paced mode, but the **project submission and the peer‑review phase must occur while a live cohort is open**. Once the peer‑review deadline passes, you’ll see your score and the reviewers’ feedback, and the certificate will be issued.'

In [82]:
rag("Que dia é hoje?")

"I don't know."